# Feature Engineering & Data Preparation

In this notebook, we transform our raw price and volume data into a format suitable for our Machine Learning models.

This involves:
1. Stationarity Testing (ADF Test)
2. Feature Scaling (MinMaxScaler)
3. Sequence Generation for LSTM (3D Tensors)


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set paths
sys.path.append(os.path.abspath('..'))
from src.data.stationarity_tests import run_adf_test


## 1. Stationarity Testing

We use the Augmented Dickey-Fuller (ADF) test to see if our time series is stationary (mean and variance are constant over time). Machine Learning models struggle with non-stationary data.


In [ ]:
df_btc = pd.read_csv('../data/processed/btc_features.csv', index_col='timestamp', parse_dates=True)
price = df_btc['price']

print('Testing Raw Price...')
res_raw = run_adf_test(price, 'BTC Raw Price')

print('\nTesting Differenced Price...')
res_diff = run_adf_test(price.diff().dropna(), 'BTC Differenced Price')


## 2. Feature Scaling & Sequence Generation

LSTMs require input data to be scaled (usually between 0 and 1) and formatted as 3D sequences: `(samples, time_steps, features)`.


In [ ]:
from src.data.preprocessing import create_sequences
import joblib

# Load scaled data
train_df = pd.read_csv('../data/processed/btc_train_scaled.csv', index_col='timestamp', parse_dates=True)

print('Raw tabular data shape:', train_df.shape)

# Generate sequences
SEQ_LEN = 30
X, y = create_sequences(train_df, seq_len=SEQ_LEN)

print(f'Generated 3D Sequence Shape (X): {X.shape}')
print(f'Generated Target Shape (y): {y.shape}')
print(f'This means we have {X.shape[0]} samples, each looking back {X.shape[1]} days, with {X.shape[2]} features.')
